You are tasked with locating 3 ambulances in Stillwater. There are 5 sites under consideration. The 3 ambulances must serve the 6 landmarks below. You want to minimize the worst-case response time.

In [2]:
time_min = { (1,'Airport') : 2, (1,'Boomer Lake') : 3, (1,'Hideaway') : 10,
             (1,'Couch Park') : 15, (1,'Movies') : 5, (1,'Edmon Low') : 7,
             (2,'Airport') : 5, (2,'Boomer Lake') : 6, (2,'Hideaway') : 7,
             (2,'Couch Park') : 12, (2,'Movies') : 14, (2,'Edmon Low') : 8,
             (3,'Airport') : 9, (3,'Boomer Lake') : 8, (3,'Hideaway') : 1,
             (3,'Couch Park') : 4, (3,'Movies') : 8, (3,'Edmon Low') : 5,
             (4,'Airport') : 11, (4,'Boomer Lake') : 10, (4,'Hideaway') : 4,
             (4,'Couch Park') : 3, (4,'Movies') : 9, (4,'Edmon Low') : 8,
             (5,'Airport') : 5, (5,'Boomer Lake') : 3, (5,'Hideaway') : 9,
             (5,'Couch Park') : 10, (5,'Movies') : 3, (5,'Edmon Low') : 9 }

In [3]:
k = 3
sites = { i for (i,j) in time_min.keys() }
landmarks = { j for (i,j) in time_min.keys() }
print(k,sites,landmarks)

3 {1, 2, 3, 4, 5} {'Boomer Lake', 'Hideaway', 'Couch Park', 'Edmon Low', 'Airport', 'Movies'}


In [6]:
import gurobipy as gp
from gurobipy import GRB

In [7]:
m = gp.Model()

x = m.addVars( sites, landmarks, vtype=GRB.BINARY )
y = m.addVars( sites, vtype=GRB.BINARY )
z = m.addVar()

Restricted license - for non-production use only - expires 2026-11-23


In [10]:
m.setObjective( z, GRB.MINIMIZE )

In [15]:
m.addConstr( gp.quicksum(y) == k )

m.addConstrs( gp.quicksum( x[i,j] for i in sites ) == 1 for j in landmarks )

m.addConstrs( x[i,j] <= y[i] for i in sites for j in landmarks )

m.addConstrs( gp.quicksum( time_min[i,j] * x[i,j] for i in sites ) <= z for j in landmarks )

m.update()

/var/folders/6r/g7p7808s1fschvz6srm5pyxc0000gn/T/ipykernel_13172/2712745297.py:1: DeprecationWarning: Calling quicksum on a tupledict is deprecated, use .sum() instead.
  m.addConstr( gp.quicksum(y) == k )


In [12]:
m.optimize()

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 24.4.0 24E263)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 43 rows, 36 columns and 131 nonzeros
Model fingerprint: 0xb5ee2195
Variable types: 1 continuous, 35 integer (35 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+00]
Found heuristic solution: objective 9.0000000
Presolve removed 10 rows and 9 columns
Presolve time: 0.01s
Presolved: 33 rows, 27 columns, 86 nonzeros
Variable types: 0 continuous, 27 integer (26 binary)

Root relaxation: objective 5.000000e+00, 13 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0       5.0000000    5.00000  0.00%     -    

In [14]:
centers = [ i for i in sites if y[i].x > 0.5 ]
print("Site ambulances at these places:",centers)

print("The worst-case response time is",m.objVal)

Site ambulances at these places: [1, 3, 4]
The worst-case response time is 5.0
